# 01 — Data Collection
## London Safety Analysis · Kudzanayi Shepherd Mhlanga

This notebook documents how crime data is collected from the [data.police.uk](https://data.police.uk) public API.

> **Note:** `data/raw/crime_data_raw.csv` is already present in this repo.  
> To refresh with live data run: `python src/collect_data.py`

### What is data.police.uk?
- Free public API from the UK Home Office — no key required
- Street-level crime data for all police forces in England & Wales
- Monthly records going back to 2010
- 15 crime categories: violence, burglary, drugs, ASB, and more

---


In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

sys.path.insert(0, str(Path.cwd().parent))
from src.config import DATA_RAW, BOROUGH_CENTROIDS, POLICE_API_BASE

plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded.")
print(f"Data directory: {DATA_RAW}")


## 1. The API structure

We call the crimes-street endpoint once per borough per month:

```
GET https://data.police.uk/api/crimes-street/all-crime?lat=51.51&lng=-0.12&date=2024-01
```

**33 boroughs × 24 months = 792 API calls total.**  
The `src/police_api.py` utility handles retries and rate limiting.


In [ ]:
import urllib.parse

borough = "Tower Hamlets"
lat, lon = BOROUGH_CENTROIDS[borough]
date = "2024-01"
params = {"lat": lat, "lng": lon, "date": date}
url = f"{POLICE_API_BASE}/crimes-street/all-crime?{urllib.parse.urlencode(params)}"

print(f"Borough  : {borough}")
print(f"Centroid : {lat}N, {lon}E")
print(f"Month    : {date}")
print(f"API call : {url}")


## 2. Load the collected data

In [ ]:
df = pd.read_csv(DATA_RAW / "crime_data_raw.csv", parse_dates=["month"])

print(f"Shape       : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Boroughs    : {df['borough'].nunique()}")
print(f"Date range  : {df['month'].min().strftime('%b %Y')} to {df['month'].max().strftime('%b %Y')}")
print(f"Crime types : {df['crime_type'].nunique()}")
print()
df.head()


In [ ]:
print("Null counts:")
print(df.isnull().sum())
print()
print("Column dtypes:")
print(df.dtypes)


## 3. Borough coverage

In [ ]:
borough_counts = df.groupby("borough").size().sort_values(ascending=True).rename("total")

fig, ax = plt.subplots(figsize=(10, 9))
colours = plt.cm.RdYlGn(np.linspace(0.85, 0.15, len(borough_counts)))
borough_counts.plot(kind="barh", ax=ax, color=colours[::-1], edgecolor="white", lw=0.5)
ax.set_xlabel("Total crime incidents (24 months)", fontsize=11)
ax.set_title("Crime Incidents per London Borough  |  2024-2025", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig(Path.cwd().parent / "reports/figures/01_borough_totals.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Crime type breakdown

In [ ]:
crime_counts = df["crime_type"].value_counts()
crime_pct    = (crime_counts / len(df) * 100).round(1)

print(f"{'Crime type':<45} {'Count':>8}  {'%':>5}")
print("-" * 65)
for ct, cnt, pct in zip(crime_counts.index, crime_counts, crime_pct):
    bar = "=" * max(1, int(pct * 1.5))
    print(f"  {ct:<43} {cnt:>8,}  {pct:>5.1f}%  {bar}")


## 5. Monthly trend

In [ ]:
monthly = df.groupby("month").size()

fig, ax = plt.subplots(figsize=(12, 4))
monthly.plot(ax=ax, color="#2980b9", lw=2, marker="o", ms=4)
ax.fill_between(monthly.index, monthly.values, alpha=0.12, color="#2980b9")
ax.set_title("Total Crime Incidents per Month -- All London Boroughs", fontweight="bold")
ax.set_ylabel("Crime incidents")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig(Path.cwd().parent / "reports/figures/01_monthly_trend.png", dpi=150, bbox_inches="tight")
plt.show()
print("Data collection notebook complete.")
print(f"Total records: {len(df):,}")
